In [24]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [25]:
project_root = Path.cwd().parents[1]

bronze_path = project_root / ".." / "data" / "bronze" / "bronze_burvot_t1_rhone_69.csv"
silver_bureau_path = project_root / ".." / "data" / "silver" / "silver_burvot_t1_rhone_69_bureau.csv"
silver_candidate_path = project_root / ".." / "data" / "silver" / "silver_burvot_t1_rhone_69_candidate.csv"

# Read bronze
df_bronze = pd.read_csv(bronze_path, sep=";", dtype=str, encoding="utf-8")
df_bronze.columns = [c.strip() for c in df_bronze.columns]

In [26]:
# Keep code columns as strings to preserve leading zeros
base_dtypes = {
    "Code du département": "string",
    "Libellé du département": "string",
    "Code de la circonscription": "string",
    "Libellé de la circonscription": "string",
    "Code de la commune": "string",
    "Libellé de la commune": "string",
    "Code du b.vote": "string",   # identifier, not a measure
    "Inscrits": "Int64",
    "Abstentions": "Int64",
    "% Abs/Ins": "Float64",
    "Votants": "Int64",
    "% Vot/Ins": "Float64",
    "Blancs": "Int64",
    "% Blancs/Ins": "Float64",
    "% Blancs/Vot": "Float64",
    "Nuls": "Int64",
    "% Nuls/Ins": "Float64",
    "% Nuls/Vot": "Float64",
    "Exprimés": "Int64",
    "% Exp/Ins": "Float64",
    "% Exp/Vot": "Float64",
}

# Candidate columns repeat with suffixes: '', '.1', '.2', ..., '.11'
candidate_dtypes = {}
for suffix in [""] + [f".{i}" for i in range(1, 12)]:
    candidate_dtypes.update({
        f"N°Panneau{suffix}": "Int64",
        f"Sexe{suffix}": "string",
        f"Nom{suffix}": "string",
        f"Prénom{suffix}": "string",
        f"Voix{suffix}": "Int64",
        f"% Voix/Ins{suffix}": "Float64",
        f"% Voix/Exp{suffix}": "Float64",
    })

meta_dtypes = {
    "extraction_source_url": "string",
    "ingestion_timestamp": "string",
    "source_file_name": "string",
}

dtype_map = {**base_dtypes, **candidate_dtypes, **meta_dtypes}

df = df_bronze.copy()

for col, dtype in dtype_map.items():
    if col not in df.columns:
        continue

    if dtype in {"Int64", "Float64"}:
        df[col] = pd.to_numeric(
            df[col].astype("string").str.replace(",", ".", regex=False),
            errors="coerce",
        ).astype(dtype)
    else:
        df[col] = df[col].astype(dtype).str.strip()

# Remove trailing ".0" from bureau code (artifact of float parsing in source)
df["Code du b.vote"] = df["Code du b.vote"].str.replace(r"\.0$", "", regex=True)

In [27]:
bureau_cols = [
    "Code du département",
    "Libellé du département",
    "Code de la circonscription",
    "Libellé de la circonscription",
    "Code de la commune",
    "Libellé de la commune",
    "Code du b.vote",
    "Inscrits",
    "Abstentions",
    "% Abs/Ins",
    "Votants",
    "% Vot/Ins",
    "Blancs",
    "% Blancs/Ins",
    "% Blancs/Vot",
    "Nuls",
    "% Nuls/Ins",
    "% Nuls/Vot",
    "Exprimés",
    "% Exp/Ins",
    "% Exp/Vot",
    "extraction_source_url",
    "ingestion_timestamp",
    "source_file_name",
]

df_bureau = df[bureau_cols].copy()

df_bureau.insert(0, "bureau_id", (
    df_bureau["Code du département"].fillna("")
    + "_" + df_bureau["Code de la circonscription"].fillna("")
    + "_" + df_bureau["Code de la commune"].fillna("")
    + "_" + df_bureau["Code du b.vote"].fillna("")
))

df_bureau = df_bureau.drop_duplicates(subset=["bureau_id"]).reset_index(drop=True)

print("Bureau shape:", df_bureau.shape)
display(df_bureau.head())

Bureau shape: (1317, 25)


,bureau_id,Code du département,Libellé du département,Code de la circonscription,Libellé de la circonscription,Code de la commune,Libellé de la commune,Code du b.vote,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,extraction_source_url,ingestion_timestamp,source_file_name
0,69_08_001_0001,69,Rhône,08,8ème circonscription,001,Affoux,0001,316,60,18.99,256,81.01,4,1.27,1.56,2,0.63,0.78,250,79.11,97.66,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
1,69_09_002_0001,69,Rhône,09,9ème circonscription,002,Aigueperse,0001,217,58,26.73,159,73.27,5,2.3,3.14,1,0.46,0.63,153,70.51,96.23,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
2,69_05_003_0001,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0001,999,194,19.42,805,80.58,11,1.1,1.37,6,0.6,0.75,788,78.88,97.89,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
3,69_05_003_0002,69,Rhône,05,5ème circonscription,003,Albigny-sur-Saône,0002,862,224,25.99,638,74.01,6,0.7,0.94,4,0.46,0.63,628,72.85,98.43,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
4,69_09_004_0001,69,Rhône,09,9ème circonscription,004,Alix,0001,470,73,15.53,397,84.47,6,1.28,1.51,2,0.43,0.5,389,82.77,97.98,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx


In [28]:
candidate_rows = []

# 12 candidate groups: '', '.1', '.2', ..., '.11'
for suffix in [""] + [f".{i}" for i in range(1, 12)]:
    cols_needed = [
        f"N°Panneau{suffix}", f"Sexe{suffix}", f"Nom{suffix}",
        f"Prénom{suffix}", f"Voix{suffix}", f"% Voix/Ins{suffix}", f"% Voix/Exp{suffix}",
    ]
    if not all(c in df.columns for c in cols_needed):
        continue

    temp = pd.DataFrame({
        "bureau_id": (
            df["Code du département"].fillna("").astype("string")
            + "_" + df["Code de la circonscription"].fillna("").astype("string")
            + "_" + df["Code de la commune"].fillna("").astype("string")
            + "_" + df["Code du b.vote"].fillna("").astype("string")
        ),
        "numero_panneau": df[f"N°Panneau{suffix}"],
        "sexe": df[f"Sexe{suffix}"],
        "nom": df[f"Nom{suffix}"],
        "prenom": df[f"Prénom{suffix}"],
        "voix": df[f"Voix{suffix}"],
        "pct_voix_ins": df[f"% Voix/Ins{suffix}"],
        "pct_voix_exp": df[f"% Voix/Exp{suffix}"],
        "extraction_source_url": df["extraction_source_url"],
        "ingestion_timestamp": df["ingestion_timestamp"],
        "source_file_name": df["source_file_name"],
    })
    candidate_rows.append(temp)

df_candidate = pd.concat(candidate_rows, ignore_index=True)

# Drop rows with no candidate info
df_candidate = df_candidate[
    ~(df_candidate["numero_panneau"].isna() & df_candidate["nom"].isna())
].reset_index(drop=True)

print("Candidate shape:", df_candidate.shape)
display(df_candidate.head(15))

Candidate shape: (15804, 11)


,bureau_id,numero_panneau,sexe,nom,prenom,voix,pct_voix_ins,pct_voix_exp,extraction_source_url,ingestion_timestamp,source_file_name
0,69_08_001_0001,1,F,ARTHAUD,Nathalie,0,0.0,0.0,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
1,69_09_002_0001,1,F,ARTHAUD,Nathalie,1,0.46,0.65,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
2,69_05_003_0001,1,F,ARTHAUD,Nathalie,2,0.2,0.25,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
3,69_05_003_0002,1,F,ARTHAUD,Nathalie,2,0.23,0.32,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
4,69_09_004_0001,1,F,ARTHAUD,Nathalie,1,0.21,0.26,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
5,69_09_005_0001,1,F,ARTHAUD,Nathalie,1,0.25,0.29,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
6,69_08_006_0001,1,F,ARTHAUD,Nathalie,3,0.35,0.49,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
7,69_08_006_0002,1,F,ARTHAUD,Nathalie,4,0.42,0.56,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
8,69_08_006_0003,1,F,ARTHAUD,Nathalie,4,0.56,0.74,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx
9,69_08_006_0004,1,F,ARTHAUD,Nathalie,6,0.63,0.81,https://static.data.gouv.fr/resources/election...,2026-03-07T20:42:59.836472,burvot_t1_france_entiere.xlsx


In [29]:
df_bureau.to_csv(silver_bureau_path, sep=";", index=False, encoding="utf-8")
df_candidate.to_csv(silver_candidate_path, sep=";", index=False, encoding="utf-8")

print(f"Wrote bureau silver:    {silver_bureau_path}")
print(f"Wrote candidate silver: {silver_candidate_path}")

Wrote bureau silver:    /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/../data/silver/silver_burvot_t1_rhone_69_bureau.csv
Wrote candidate silver: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/../data/silver/silver_burvot_t1_rhone_69_candidate.csv
